# Module 5.1: Dedicated Tier-0 Raspberry Pi Architecture & Pi-hole v6 DNS Server

## Homelab Expansion (Chapter 2) - Step 5: Core Raspberry Pi Infrastructure Services

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Articulate why dedicating the Raspberry Pi (`homelab-pi`) exclusively as a **Tier-0 Network & Observability Core**—with zero heavy apps or databases—maximizes reliability across your entire homelab.
2. Understand **Fate Sharing / Blast Radius Isolation**: why network-critical services like DNS, VPN subnet routing, and uptime monitoring must never share a host with heavy GPU/media workloads.
3. Deploy **Pi-hole v6** via Docker Compose on `homelab-pi` using modern `FTLCONF_` configuration variables and MicroSD-friendly query database retention (`~/homelab/data/pihole`).
4. Configure local LAN DNS resolution (`homelab-pi.lan`, `homelab-gpu.lan`) and **Tailscale Global Split-DNS** so every phone and laptop on your Tailnet gets automatic ad-blocking anywhere in the world.

## 5.1.1 Conceptual Background: Why a Dedicated "Tier-0 Network & Monitoring" Pi Beats Mixing Workloads

Now that the entire Immich stack and your WD Blue SN5000 NVMe SSD live inside your powerful 6–8 core ROG Zephyrus (`homelab-gpu`), a natural question arises: *"Should I just retire the Raspberry Pi and move DNS and monitoring onto the laptop too?"*

**Absolutely not!** In professional systems engineering, dedicating an independent, ultra-low-power node strictly to **Control-Plane / Tier-0 Network Infrastructure (`homelab-pi`)** while consolidating **Applications, Storage & GPU Compute (`homelab-gpu`)** is the cleanest architectural boundary you can build:

### 1. Why the Raspberry Pi Is the Ideal Dedicated "Tier-0 Core" Appliance
1. **Zero Fate Sharing (True Blast-Radius Isolation):** Imagine you are upgrading Immich, updating NVIDIA kernel drivers (`nvidia.ko`), testing a local LLM (`Ollama`), or running a heavy 4K NVENC transcode batch on `homelab-gpu`—requiring a host reboot. Because `homelab-pi` runs **zero heavy apps**, your home's primary DNS server (`Pi-hole`), remote VPN gateway (`Tailscale`), and monitoring (`Uptime Kuma`) stay **100% unaffected**.
2. **Ultra-Low ~4.5W Fanless Footprint & Tiny Memory Profile:** Without Immich, `homelab-pi` has **no external USB drive**, zero moving parts, and uses under **~400 MB of its 4 GB RAM**—meaning it needs no disk swapfile and runs for hours on your UPS during an extended outage.
3. **MicroSD Longevity Without an External SSD:** Because `homelab-pi` only runs lightweight control-plane services (Pi-hole v6 + Uptime Kuma), storing their tiny SQLite configs locally in `~/homelab/data/`—paired with Pi-hole v6's `FTLCONF_database_maxDBdays=30` and `FTLCONF_database_DBinterval=5.0` (flushing stats every 5 minutes instead of every minute)—keeps MicroSD write amplification negligible while Chapter 3 backs them up nightly to `UNAS 2`.

### 2. The Definitive Multi-Node Service Allocation Matrix

| Service | Assigned Node | Footprint | Why It Belongs Here |
| :--- | :---: | :---: | :--- |
| **Pi-hole v6 (Network-Wide DNS Sinkhole & Local DNS)** | **Node 1 (`homelab-pi`)** | ~65 MB RAM | Tier-0 network dependency. Must stay online 24/7/365 with sub-millisecond response even when `homelab-gpu` is rebooting or under full GPU load. |
| **Tailscale Subnet Router & Global DNS Endpoint** | **Node 1 (`homelab-pi`)** | ~35 MB RAM | Always-on zero-port entry point into your home LAN (`UNAS 2` & switch in Chapter 3). |
| **Uptime Kuma (Independent Cluster Monitor)** | **Node 1 (`homelab-pi`)** | ~110 MB RAM | An external observer must never live on the app server it is watching—if `homelab-gpu` hangs or reboots, `homelab-pi` alerts you immediately! |
| **Full Immich Stack (`server`, `postgres`, `redis`, `ml-cuda`)** | **Node 2 (`homelab-gpu`)** | ~2.5 GB RAM + VRAM | Internal Slot-2 WD Blue SN5000 NVMe + NVIDIA **CUDA** ML + **NVENC** hardware video transcoding all on one high-throughput host. |
| **Chapter 3 Personal Cloud Apps (UniFi, Paperless-ngx, Mail-Archiver, FileBrowser)** | **Node 2 (`homelab-gpu`)** | ~3–5 GB RAM | Heavy JVM/MongoDB, OCR, and PostgreSQL workloads belong on `x86_64` with 16–32 GB DDR4 RAM. |
| **Optional Future AI / Media (Ollama, Jellyfin NVENC)** | **Node 2 (`homelab-gpu`)** | 4–8 GB VRAM | Runs directly on the NVIDIA GPU without ever touching Tier-0 DNS or monitoring. |

## 5.1.2 Step-by-Step Laboratory Guide: Installing Pi-hole v6 on `homelab-pi`

### Task 5.1.1: Verify Port 53 Availability on `homelab-pi`
DNS operates on **TCP and UDP Port `53`**. Before deploying Pi-hole on `homelab-pi`, connect via `ssh homelab-pi` and verify that no local stub resolver (such as `systemd-resolved`) is occupying port `53`:
```bash
# Check if anything is currently listening on port 53 on homelab-pi
sudo ss -tulpn | grep ":53 " || echo "Port 53 is free and ready for Pi-hole!"
```
*(Note: If `systemd-resolved` is listed on `127.0.0.53:53`, disable its stub listener with `sudo sed -r -i.orig 's/#?DNSStubListener=yes/DNSStubListener=no/g' /etc/systemd/resolved.conf && sudo systemctl restart systemd-resolved`).*

### Task 5.1.2: Create the Pi-hole v6 Docker Compose Stack on `homelab-pi`
Create clean, self-contained local directories under `~/homelab/` on `homelab-pi` (making Chapter 3 nightly backups to `UNAS 2` a single tidy `rsync` of `~/homelab/`!):

```bash
# 1. Create stack directory and persistent local data directories on homelab-pi
mkdir -p ~/homelab/stacks/pihole
mkdir -p ~/homelab/data/pihole/etc-pihole
mkdir -p ~/homelab/data/pihole/etc-dnsmasq.d
cd ~/homelab/stacks/pihole
```

Create a `.env` file to hold your Pi-hole v6 administrator password (replace the placeholder with your own strong password locally):
```bash
cat << 'EOT' > ~/homelab/stacks/pihole/.env
PIHOLE_PASSWORD=[Write your password here]
EOT
chmod 600 ~/homelab/stacks/pihole/.env
```

Now create the **Pi-hole v6** `docker-compose.yml` file (note that Pi-hole v6 unified its web server inside `pihole-FTL` and uses `FTLCONF_` variables—including `FTLCONF_database_DBinterval=5.0` and `FTLCONF_database_maxDBdays=30` to minimize MicroSD write wear!):
```bash
cat << 'EOT' > ~/homelab/stacks/pihole/docker-compose.yml
name: pihole

services:
  pihole:
    container_name: pihole
    image: pihole/pihole:latest
    ports:
      - "53:53/tcp"
      - "53:53/udp"
      - "8080:80/tcp"
    environment:
      TZ: "America/New_York"
      # Pi-hole v6 unified FTL configuration variables:
      FTLCONF_webserver_api_password: "${PIHOLE_PASSWORD}"
      FTLCONF_dns_upstreams: "1.1.1.1;1.0.0.1;9.9.9.9"
      # 'all' permits queries from both local LAN (10.0.0.x) and Tailscale (100.x.y.z)
      FTLCONF_dns_listeningMode: "all"
      # MicroSD longevity tuning: flush SQLite query log every 5 min and keep 30 days
      FTLCONF_database_DBinterval: "5.0"
      FTLCONF_database_maxDBdays: "30"
    volumes:
      - /home/pi/homelab/data/pihole/etc-pihole:/etc/pihole
      - /home/pi/homelab/data/pihole/etc-dnsmasq.d:/etc/dnsmasq.d
    cap_add:
      - NET_ADMIN
    restart: unless-stopped
EOT
```
*(Why port `8080:80`? Mapping the Pi-hole web UI to host port `8080` keeps port `80`/`443` free on `homelab-pi` in case you later add a reverse proxy like Caddy or Nginx).*

Launch the Pi-hole v6 stack:
```bash
cd ~/homelab/stacks/pihole
docker compose up -d
docker compose ps
```

### Task 5.1.3: Verify DNS Resolution, Custom Local DNS Records & Web Dashboard
1. Test DNS resolution directly against `homelab-pi` (`10.0.0.200`) from your terminal:
   ```bash
   # 1. Verify normal domain resolution works (<20ms)
   nslookup github.com 10.0.0.200

   # 2. Verify a known advertising/tracking domain is sinkholed to 0.0.0.0!
   nslookup doubleclick.net 10.0.0.200
   ```
2. Open the **Pi-hole v6 Web Admin Interface** in your browser:
   ```text
   http://10.0.0.200:8080/admin
   ```
3. Log in with your password and add **Local DNS Records** (**Settings $\rightarrow$ Local DNS Records**) for your homelab nodes so every device on your network can resolve them by clean names:
   - `homelab-pi.lan` $\rightarrow$ `10.0.0.200`
   - `homelab-gpu.lan` $\rightarrow$ `10.0.0.201`
4. **Integrate with Your Home Network & Tailscale Mesh:**
   - **Home LAN:** In your home router's DHCP settings, set the **Primary DNS Server** to `10.0.0.200` so every smart TV, laptop, and phone on home Wi-Fi uses Pi-hole automatically.
   - **Remote Tailscale Ad-Blocking Everywhere:** In the [Tailscale Admin Console $\rightarrow$ DNS tab](https://login.tailscale.com/admin/dns), click **Add nameserver $\rightarrow$ Custom**, enter your Raspberry Pi's `100.x.y.z` Tailscale IP, and toggle **Override local DNS** ON. Now your smartphone blocks ads over 5G cellular data anywhere in the world through your Raspberry Pi!

### 🖼️ Task 5.1.3 Verification: Pi-hole v6 Dashboard & Sinkholed Query Verification

**Verification Objective:** Capture a screenshot of your Pi-hole v6 Web Dashboard (`http://10.0.0.200:8080/admin`) showing active DNS queries blocked and your terminal `nslookup doubleclick.net 10.0.0.200` returning `0.0.0.0`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Pi-hole v6 deployed on `homelab-pi` with MicroSD-friendly SQLite retention stored under `~/homelab/data/pihole`.
- [ ] `nslookup doubleclick.net 10.0.0.200` returns `0.0.0.0`.
- [ ] Local DNS records created for `homelab-pi.lan` (`10.0.0.200`) and `homelab-gpu.lan` (`10.0.0.201`).
- [ ] Tailscale / Router DNS pointed to `homelab-pi` for automatic ad-blocking.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 5.2: Unified Homelab Monitoring with Uptime Kuma](./05_2_homelab_monitoring_uptime_kuma.ipynb).